# Experiment Tracking and Management Framework

This notebook implements a comprehensive experiment tracking system including:
- Experiment versioning and reproducibility
- Hyperparameter tracking
- Metric logging and visualization
- Model checkpointing and versioning
- Experiment comparison and analysis
- Integration with MLflow and Weights & Biases

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Experiment tracking
import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient
import wandb

# Utilities
import json
import yaml
import pickle
import hashlib
import uuid
from pathlib import Path
from datetime import datetime
import git
from typing import Dict, List, Optional, Any, Union
from dataclasses import dataclass, field, asdict

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings

warnings.filterwarnings("ignore")

# Set style
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

## 1. Experiment Configuration

In [ ]:
@dataclass
class ExperimentConfig:
    """Configuration for an experiment."""

    # Experiment metadata
    name: str = "default_experiment"
    description: str = ""
    tags: Dict[str, str] = field(default_factory=dict)

    # Model configuration
    model_type: str = "neural_network"
    model_params: Dict[str, Any] = field(default_factory=dict)

    # Training configuration
    batch_size: int = 32
    learning_rate: float = 0.001
    epochs: int = 100
    optimizer: str = "adam"
    loss_function: str = "cross_entropy"

    # Data configuration
    dataset: str = "default"
    data_params: Dict[str, Any] = field(default_factory=dict)

    # Environment
    seed: int = 42
    device: str = "auto"

    # Tracking
    track_gradients: bool = False
    log_frequency: int = 10
    checkpoint_frequency: int = 5

    def to_dict(self) -> Dict:
        """Convert config to dictionary."""
        return asdict(self)

    def to_yaml(self, filepath: str):
        """Save config to YAML file."""
        with open(filepath, "w") as f:
            yaml.dump(self.to_dict(), f)

    @classmethod
    def from_yaml(cls, filepath: str):
        """Load config from YAML file."""
        with open(filepath, "r") as f:
            config_dict = yaml.load(f, Loader=yaml.FullLoader)
        return cls(**config_dict)

    def get_hash(self) -> str:
        """Get unique hash for this configuration."""
        config_str = json.dumps(self.to_dict(), sort_keys=True)
        return hashlib.md5(config_str.encode()).hexdigest()[:8]

## 2. Experiment Tracker

In [ ]:
class ExperimentTracker:
    """Base class for experiment tracking."""

    def __init__(
        self,
        config: ExperimentConfig,
        backend: str = "local",
        project_name: str = "deep_learning_experiments",
    ):
        """
        Initialize experiment tracker.

        Parameters:
        -----------
        config : ExperimentConfig
            Experiment configuration
        backend : str
            Tracking backend ('local', 'mlflow', 'wandb')
        project_name : str
            Project name
        """
        self.config = config
        self.backend = backend
        self.project_name = project_name
        self.run_id = None
        self.metrics_history = {}
        self.artifacts = {}

        # Initialize backend
        self._init_backend()

    def _init_backend(self):
        """Initialize tracking backend."""
        if self.backend == "mlflow":
            mlflow.set_experiment(self.project_name)
            mlflow.start_run(run_name=self.config.name)
            self.run_id = mlflow.active_run().info.run_id

            # Log parameters
            mlflow.log_params(self.config.to_dict())

        elif self.backend == "wandb":
            wandb.init(
                project=self.project_name,
                name=self.config.name,
                config=self.config.to_dict(),
            )
            self.run_id = wandb.run.id

        else:  # local
            self.run_id = str(uuid.uuid4())
            self.run_dir = Path(f"experiments/{self.project_name}/{self.run_id}")
            self.run_dir.mkdir(parents=True, exist_ok=True)

            # Save config
            self.config.to_yaml(self.run_dir / "config.yaml")

    def log_metrics(self, metrics: Dict[str, float], step: int = None):
        """Log metrics."""
        if self.backend == "mlflow":
            mlflow.log_metrics(metrics, step=step)

        elif self.backend == "wandb":
            wandb.log(metrics, step=step)

        else:  # local
            for key, value in metrics.items():
                if key not in self.metrics_history:
                    self.metrics_history[key] = []
                self.metrics_history[key].append({"step": step, "value": value})

    def log_artifact(self, artifact_path: str, artifact_type: str = "file"):
        """Log artifact."""
        if self.backend == "mlflow":
            if artifact_type == "file":
                mlflow.log_artifact(artifact_path)
            elif artifact_type == "model":
                mlflow.pytorch.log_model(artifact_path, "model")

        elif self.backend == "wandb":
            wandb.save(artifact_path)

        else:  # local
            import shutil

            dest = self.run_dir / "artifacts" / Path(artifact_path).name
            dest.parent.mkdir(exist_ok=True)
            shutil.copy2(artifact_path, dest)

    def log_model(self, model: nn.Module, model_name: str = "model"):
        """Log model."""
        if self.backend == "mlflow":
            mlflow.pytorch.log_model(model, model_name)

        elif self.backend == "wandb":
            torch.save(model.state_dict(), f"{model_name}.pt")
            wandb.save(f"{model_name}.pt")

        else:  # local
            model_path = self.run_dir / f"{model_name}.pt"
            torch.save(model.state_dict(), model_path)

    def end_run(self):
        """End the current run."""
        if self.backend == "mlflow":
            mlflow.end_run()

        elif self.backend == "wandb":
            wandb.finish()

        else:  # local
            # Save metrics history
            metrics_file = self.run_dir / "metrics.json"
            with open(metrics_file, "w") as f:
                json.dump(self.metrics_history, f, indent=2)

## 3. Experiment Manager

In [ ]:
class ExperimentManager:
    """Manage multiple experiments."""

    def __init__(
        self,
        project_name: str = "deep_learning_experiments",
        base_dir: str = "./experiments",
    ):
        """
        Initialize experiment manager.

        Parameters:
        -----------
        project_name : str
            Project name
        base_dir : str
            Base directory for experiments
        """
        self.project_name = project_name
        self.base_dir = Path(base_dir)
        self.experiments_dir = self.base_dir / project_name
        self.experiments_dir.mkdir(parents=True, exist_ok=True)

        # Load experiment registry
        self.registry_file = self.experiments_dir / "registry.json"
        self.registry = self._load_registry()

    def _load_registry(self) -> Dict:
        """Load experiment registry."""
        if self.registry_file.exists():
            with open(self.registry_file, "r") as f:
                return json.load(f)
        return {}

    def _save_registry(self):
        """Save experiment registry."""
        with open(self.registry_file, "w") as f:
            json.dump(self.registry, f, indent=2)

    def register_experiment(
        self, config: ExperimentConfig, tracker: ExperimentTracker
    ) -> str:
        """Register a new experiment."""
        experiment_id = tracker.run_id

        # Create experiment entry
        experiment_entry = {
            "id": experiment_id,
            "name": config.name,
            "description": config.description,
            "config_hash": config.get_hash(),
            "created_at": datetime.now().isoformat(),
            "status": "running",
            "tags": config.tags,
            "backend": tracker.backend,
        }

        # Add to registry
        self.registry[experiment_id] = experiment_entry
        self._save_registry()

        return experiment_id

    def update_experiment_status(
        self, experiment_id: str, status: str, metrics: Optional[Dict] = None
    ):
        """Update experiment status."""
        if experiment_id in self.registry:
            self.registry[experiment_id]["status"] = status
            self.registry[experiment_id]["updated_at"] = datetime.now().isoformat()

            if metrics:
                self.registry[experiment_id]["final_metrics"] = metrics

            self._save_registry()

    def get_experiments(
        self, filter_tags: Optional[Dict] = None, status: Optional[str] = None
    ) -> List[Dict]:
        """Get experiments with optional filtering."""
        experiments = list(self.registry.values())

        # Filter by status
        if status:
            experiments = [e for e in experiments if e["status"] == status]

        # Filter by tags
        if filter_tags:
            experiments = [
                e
                for e in experiments
                if all(e.get("tags", {}).get(k) == v for k, v in filter_tags.items())
            ]

        return experiments

    def compare_experiments(self, experiment_ids: List[str]) -> pd.DataFrame:
        """Compare multiple experiments."""
        comparison_data = []

        for exp_id in experiment_ids:
            if exp_id in self.registry:
                exp = self.registry[exp_id]
                row = {
                    "id": exp_id[:8],
                    "name": exp["name"],
                    "status": exp["status"],
                    "created_at": exp["created_at"],
                }

                # Add final metrics if available
                if "final_metrics" in exp:
                    row.update(exp["final_metrics"])

                comparison_data.append(row)

        return pd.DataFrame(comparison_data)

    def get_best_experiment(self, metric: str, mode: str = "max") -> Dict:
        """Get best experiment based on a metric."""
        experiments = [
            e
            for e in self.registry.values()
            if "final_metrics" in e and metric in e["final_metrics"]
        ]

        if not experiments:
            return None

        if mode == "max":
            return max(experiments, key=lambda x: x["final_metrics"][metric])
        else:
            return min(experiments, key=lambda x: x["final_metrics"][metric])

## 4. Reproducibility Manager

In [ ]:
class ReproducibilityManager:
    """Ensure experiment reproducibility."""

    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.environment_info = self._capture_environment()
        self.git_info = self._capture_git_info()

    def set_seeds(self):
        """Set all random seeds for reproducibility."""
        seed = self.config.seed

        # Python random
        import random

        random.seed(seed)

        # NumPy
        np.random.seed(seed)

        # PyTorch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # PyTorch deterministic behavior
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    def _capture_environment(self) -> Dict:
        """Capture environment information."""
        import platform
        import sys

        env_info = {
            "python_version": sys.version,
            "platform": platform.platform(),
            "processor": platform.processor(),
            "pytorch_version": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
            "cudnn_version": torch.backends.cudnn.version()
            if torch.cuda.is_available()
            else None,
        }

        # Capture installed packages
        import pkg_resources

        installed_packages = {pkg.key: pkg.version for pkg in pkg_resources.working_set}
        env_info["packages"] = installed_packages

        return env_info

    def _capture_git_info(self) -> Dict:
        """Capture git repository information."""
        try:
            repo = git.Repo(search_parent_directories=True)

            git_info = {
                "commit": repo.head.object.hexsha,
                "branch": repo.active_branch.name,
                "dirty": repo.is_dirty(),
                "remote_url": list(repo.remote().urls)[0] if repo.remotes else None,
                "commit_message": repo.head.object.message.strip(),
                "commit_date": repo.head.object.committed_datetime.isoformat(),
            }

            # Get uncommitted changes
            if repo.is_dirty():
                git_info["uncommitted_changes"] = [
                    item.a_path for item in repo.index.diff(None)
                ]

            return git_info
        except:
            return {"error": "Not a git repository"}

    def save_snapshot(self, save_dir: Path):
        """Save complete snapshot for reproducibility."""
        snapshot_dir = save_dir / "reproducibility"
        snapshot_dir.mkdir(exist_ok=True)

        # Save config
        self.config.to_yaml(snapshot_dir / "config.yaml")

        # Save environment info
        with open(snapshot_dir / "environment.json", "w") as f:
            json.dump(self.environment_info, f, indent=2)

        # Save git info
        with open(snapshot_dir / "git_info.json", "w") as f:
            json.dump(self.git_info, f, indent=2)

        # Create requirements file
        with open(snapshot_dir / "requirements.txt", "w") as f:
            for pkg, version in self.environment_info["packages"].items():
                f.write(f"{pkg}=={version}\n")

    def verify_reproducibility(self, other_snapshot_dir: Path) -> Dict:
        """Verify if current environment matches a snapshot."""
        verification = {"matches": True, "differences": []}

        # Load other snapshot
        with open(other_snapshot_dir / "environment.json", "r") as f:
            other_env = json.load(f)

        # Compare Python version
        if self.environment_info["python_version"] != other_env["python_version"]:
            verification["matches"] = False
            verification["differences"].append(
                f"Python version: {self.environment_info['python_version']} vs {other_env['python_version']}"
            )

        # Compare PyTorch version
        if self.environment_info["pytorch_version"] != other_env["pytorch_version"]:
            verification["matches"] = False
            verification["differences"].append(
                f"PyTorch version: {self.environment_info['pytorch_version']} vs {other_env['pytorch_version']}"
            )

        # Compare key packages
        key_packages = ["numpy", "pandas", "scikit-learn"]
        for pkg in key_packages:
            if (
                pkg in self.environment_info["packages"]
                and pkg in other_env["packages"]
            ):
                if self.environment_info["packages"][pkg] != other_env["packages"][pkg]:
                    verification["differences"].append(
                        f"{pkg}: {self.environment_info['packages'][pkg]} vs {other_env['packages'][pkg]}"
                    )

        return verification

## 5. Experiment Runner

In [ ]:
class ExperimentRunner:
    """Run and track experiments."""

    def __init__(self, config: ExperimentConfig, backend: str = "local"):
        """
        Initialize experiment runner.

        Parameters:
        -----------
        config : ExperimentConfig
            Experiment configuration
        backend : str
            Tracking backend
        """
        self.config = config
        self.backend = backend

        # Initialize components
        self.tracker = ExperimentTracker(config, backend)
        self.manager = ExperimentManager()
        self.reproducibility = ReproducibilityManager(config)

        # Set seeds
        self.reproducibility.set_seeds()

        # Register experiment
        self.experiment_id = self.manager.register_experiment(config, self.tracker)

        # Save reproducibility snapshot
        if backend == "local":
            self.reproducibility.save_snapshot(self.tracker.run_dir)

    def train_epoch(
        self,
        model: nn.Module,
        train_loader: DataLoader,
        optimizer: torch.optim.Optimizer,
        criterion: nn.Module,
        epoch: int,
    ) -> Dict[str, float]:
        """Train for one epoch with tracking."""
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for batch_idx, (data, target) in enumerate(train_loader):
            # Move to device
            if self.config.device == "auto":
                device = "cuda" if torch.cuda.is_available() else "cpu"
            else:
                device = self.config.device

            data, target = data.to(device), target.to(device)

            # Training step
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()

            # Track gradients if enabled
            if (
                self.config.track_gradients
                and batch_idx % self.config.log_frequency == 0
            ):
                grad_norms = {}
                for name, param in model.named_parameters():
                    if param.grad is not None:
                        grad_norms[f"grad_norm/{name}"] = param.grad.norm().item()
                self.tracker.log_metrics(
                    grad_norms, step=epoch * len(train_loader) + batch_idx
                )

            optimizer.step()

            # Track metrics
            total_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            # Log batch metrics
            if batch_idx % self.config.log_frequency == 0:
                batch_metrics = {
                    "train/batch_loss": loss.item(),
                    "train/batch_acc": 100.0 * correct / total,
                }
                self.tracker.log_metrics(
                    batch_metrics, step=epoch * len(train_loader) + batch_idx
                )

        # Epoch metrics
        epoch_metrics = {
            "train/epoch_loss": total_loss / len(train_loader),
            "train/epoch_acc": 100.0 * correct / total,
        }

        return epoch_metrics

    def validate(
        self, model: nn.Module, val_loader: DataLoader, criterion: nn.Module, epoch: int
    ) -> Dict[str, float]:
        """Validate model."""
        model.eval()
        total_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for data, target in val_loader:
                # Move to device
                if self.config.device == "auto":
                    device = "cuda" if torch.cuda.is_available() else "cpu"
                else:
                    device = self.config.device

                data, target = data.to(device), target.to(device)

                # Validation step
                output = model(data)
                loss = criterion(output, target)

                total_loss += loss.item()
                _, predicted = output.max(1)
                total += target.size(0)
                correct += predicted.eq(target).sum().item()

        # Validation metrics
        val_metrics = {
            "val/loss": total_loss / len(val_loader),
            "val/acc": 100.0 * correct / total,
        }

        return val_metrics

    def run(
        self,
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        optimizer: torch.optim.Optimizer,
        criterion: nn.Module,
    ):
        """Run complete experiment."""
        best_val_acc = 0

        try:
            for epoch in range(self.config.epochs):
                # Training
                train_metrics = self.train_epoch(
                    model, train_loader, optimizer, criterion, epoch
                )

                # Validation
                val_metrics = self.validate(model, val_loader, criterion, epoch)

                # Log metrics
                all_metrics = {**train_metrics, **val_metrics}
                self.tracker.log_metrics(all_metrics, step=epoch)

                # Print progress
                print(f"Epoch {epoch + 1}/{self.config.epochs}:")
                print(
                    f"  Train Loss: {train_metrics['train/epoch_loss']:.4f}, "
                    f"Train Acc: {train_metrics['train/epoch_acc']:.2f}%"
                )
                print(
                    f"  Val Loss: {val_metrics['val/loss']:.4f}, "
                    f"Val Acc: {val_metrics['val/acc']:.2f}%"
                )

                # Save checkpoint
                if epoch % self.config.checkpoint_frequency == 0:
                    self.tracker.log_model(model, f"model_epoch_{epoch}")

                # Track best model
                if val_metrics["val/acc"] > best_val_acc:
                    best_val_acc = val_metrics["val/acc"]
                    self.tracker.log_model(model, "best_model")

            # Final metrics
            final_metrics = {
                "best_val_acc": best_val_acc,
                "final_train_loss": train_metrics["train/epoch_loss"],
                "final_val_loss": val_metrics["val/loss"],
            }

            # Update experiment status
            self.manager.update_experiment_status(
                self.experiment_id, "completed", final_metrics
            )

        except Exception as e:
            # Handle errors
            print(f"Experiment failed: {e}")
            self.manager.update_experiment_status(self.experiment_id, "failed")
            raise

        finally:
            # End tracking
            self.tracker.end_run()

        return final_metrics

## 6. Experiment Analysis

In [ ]:
class ExperimentAnalyzer:
    """Analyze and visualize experiment results."""

    def __init__(self, manager: ExperimentManager):
        self.manager = manager

    def plot_metrics_comparison(
        self, experiment_ids: List[str], metrics: List[str] = ["val/acc", "val/loss"]
    ):
        """Plot metrics comparison across experiments."""
        fig = make_subplots(rows=len(metrics), cols=1, subplot_titles=metrics)

        for exp_id in experiment_ids:
            exp_dir = self.manager.experiments_dir / exp_id
            metrics_file = exp_dir / "metrics.json"

            if metrics_file.exists():
                with open(metrics_file, "r") as f:
                    metrics_data = json.load(f)

                exp_name = self.manager.registry[exp_id]["name"]

                for i, metric in enumerate(metrics, 1):
                    if metric in metrics_data:
                        values = [m["value"] for m in metrics_data[metric]]
                        steps = [m["step"] for m in metrics_data[metric]]

                        fig.add_trace(
                            go.Scatter(x=steps, y=values, name=exp_name), row=i, col=1
                        )

        fig.update_layout(height=300 * len(metrics), showlegend=True)
        return fig

    def plot_hyperparameter_importance(self, metric: str = "best_val_acc"):
        """Plot hyperparameter importance."""
        # Get completed experiments
        experiments = self.manager.get_experiments(status="completed")

        if not experiments:
            print("No completed experiments found")
            return

        # Extract hyperparameters and metrics
        data = []
        for exp in experiments:
            if "final_metrics" in exp and metric in exp["final_metrics"]:
                # Load config
                exp_dir = self.manager.experiments_dir / exp["id"]
                config_file = exp_dir / "config.yaml"

                if config_file.exists():
                    with open(config_file, "r") as f:
                        config = yaml.load(f, Loader=yaml.FullLoader)

                    row = {
                        "learning_rate": config["learning_rate"],
                        "batch_size": config["batch_size"],
                        "optimizer": config["optimizer"],
                        metric: exp["final_metrics"][metric],
                    }
                    data.append(row)

        if not data:
            print(f"No experiments with metric {metric} found")
            return

        df = pd.DataFrame(data)

        # Create subplots
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Learning rate vs metric
        axes[0].scatter(df["learning_rate"], df[metric], alpha=0.6)
        axes[0].set_xlabel("Learning Rate")
        axes[0].set_ylabel(metric)
        axes[0].set_xscale("log")
        axes[0].set_title("Learning Rate Impact")

        # Batch size vs metric
        axes[1].scatter(df["batch_size"], df[metric], alpha=0.6)
        axes[1].set_xlabel("Batch Size")
        axes[1].set_ylabel(metric)
        axes[1].set_title("Batch Size Impact")

        # Optimizer comparison
        optimizer_stats = df.groupby("optimizer")[metric].agg(["mean", "std"])
        axes[2].bar(
            optimizer_stats.index,
            optimizer_stats["mean"],
            yerr=optimizer_stats["std"],
            capsize=5,
        )
        axes[2].set_xlabel("Optimizer")
        axes[2].set_ylabel(f"Mean {metric}")
        axes[2].set_title("Optimizer Comparison")

        plt.tight_layout()
        return fig

    def generate_report(self, experiment_id: str) -> str:
        """Generate markdown report for an experiment."""
        if experiment_id not in self.manager.registry:
            return "Experiment not found"

        exp = self.manager.registry[experiment_id]
        exp_dir = self.manager.experiments_dir / experiment_id

        report = f"# Experiment Report: {exp['name']}\n\n"
        report += f"**ID:** {experiment_id}\n"
        report += f"**Status:** {exp['status']}\n"
        report += f"**Created:** {exp['created_at']}\n\n"

        # Configuration
        report += "## Configuration\n"
        config_file = exp_dir / "config.yaml"
        if config_file.exists():
            with open(config_file, "r") as f:
                config = yaml.load(f, Loader=yaml.FullLoader)

            report += "```yaml\n"
            report += yaml.dump(config, default_flow_style=False)
            report += "```\n\n"

        # Results
        if "final_metrics" in exp:
            report += "## Final Results\n"
            for metric, value in exp["final_metrics"].items():
                report += f"- **{metric}:** {value:.4f}\n"
            report += "\n"

        # Reproducibility
        report += "## Reproducibility\n"
        git_file = exp_dir / "reproducibility" / "git_info.json"
        if git_file.exists():
            with open(git_file, "r") as f:
                git_info = json.load(f)

            report += f"- **Git Commit:** {git_info.get('commit', 'N/A')[:8]}\n"
            report += f"- **Git Branch:** {git_info.get('branch', 'N/A')}\n"
            report += f"- **Repository Clean:** {not git_info.get('dirty', True)}\n"

        return report

## 7. Example Usage

In [ ]:
# Create a simple model for demonstration
class SimpleNet(nn.Module):
    def __init__(self, input_size=784, hidden_size=128, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x


# Create synthetic dataset
from torch.utils.data import TensorDataset

# Generate random data
X_train = torch.randn(1000, 1, 28, 28)
y_train = torch.randint(0, 10, (1000,))
X_val = torch.randn(200, 1, 28, 28)
y_val = torch.randint(0, 10, (200,))

# Create datasets
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Dataset created:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")

In [ ]:
# Configure experiment
config = ExperimentConfig(
    name="mnist_classification_v1",
    description="Simple MNIST classification with MLP",
    tags={"model": "mlp", "dataset": "synthetic"},
    model_type="simple_net",
    model_params={"input_size": 784, "hidden_size": 128, "num_classes": 10},
    batch_size=32,
    learning_rate=0.001,
    epochs=5,  # Reduced for demo
    optimizer="adam",
    seed=42,
    track_gradients=True,
    checkpoint_frequency=2,
)

print("Experiment configuration:")
print(f"  Name: {config.name}")
print(f"  Learning rate: {config.learning_rate}")
print(f"  Batch size: {config.batch_size}")
print(f"  Epochs: {config.epochs}")

In [ ]:
# Initialize experiment runner
runner = ExperimentRunner(config, backend="local")

# Create model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SimpleNet().to(device)

# Setup optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
criterion = nn.CrossEntropyLoss()

print(f"\nStarting experiment: {runner.experiment_id[:8]}")
print(f"Backend: {runner.backend}")
print(f"Device: {device}")

In [ ]:
# Run experiment
final_metrics = runner.run(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
)

print("\nExperiment completed!")
print("Final metrics:")
for metric, value in final_metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Analyze results
analyzer = ExperimentAnalyzer(runner.manager)

# Generate report
report = analyzer.generate_report(runner.experiment_id)
print(report)

# Get all experiments
all_experiments = runner.manager.get_experiments()
print(f"\nTotal experiments: {len(all_experiments)}")

# Compare experiments
if len(all_experiments) > 0:
    comparison_df = runner.manager.compare_experiments(
        [exp["id"] for exp in all_experiments[-3:]]
    )
    print("\nRecent experiments comparison:")
    print(comparison_df)

## Summary

This notebook provides a comprehensive experiment tracking and management framework with:

### Key Features:
1. **Experiment Configuration**: Structured configuration management with YAML support
2. **Multi-Backend Tracking**: Support for local, MLflow, and Weights & Biases
3. **Reproducibility**: Complete environment and git tracking
4. **Experiment Management**: Registry, comparison, and analysis tools
5. **Automated Tracking**: Metrics, gradients, and artifact logging

### Components:
- `ExperimentConfig`: Configuration dataclass
- `ExperimentTracker`: Multi-backend tracking
- `ExperimentManager`: Experiment registry and comparison
- `ReproducibilityManager`: Environment and code versioning
- `ExperimentRunner`: Complete training pipeline
- `ExperimentAnalyzer`: Analysis and visualization

### Benefits:
- **Reproducible Research**: Full environment and code tracking
- **Experiment Comparison**: Easy comparison across runs
- **Automated Logging**: Comprehensive metric and artifact tracking
- **Flexible Backend**: Support for multiple tracking systems

The framework ensures that all experiments are tracked, reproducible, and easily comparable for effective deep learning research and development.